In [1]:
# Configuración inicial del entorno
import sys
import os
from pathlib import Path

print("🔄 Configurando entorno...")

# Obtener el directorio raíz del proyecto
notebook_dir = Path().resolve()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    # Buscar el directorio con pyproject.toml
    project_root = notebook_dir
    while project_root != project_root.parent:
        if (project_root / "pyproject.toml").exists():
            break
        project_root = project_root.parent

print(f"📁 Directorio raíz: {project_root}")

# Cambiar al directorio raíz
os.chdir(project_root)
print(f"✓ Directorio de trabajo: {os.getcwd()}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar pyproject.toml
if (project_root / "pyproject.toml").exists():
    print("✓ pyproject.toml encontrado")
else:
    raise FileNotFoundError(f"pyproject.toml no encontrado en {project_root}")

print("✅ Configuración completada\n")


🔄 Configurando entorno...
📁 Directorio raíz: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ Directorio de trabajo: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ pyproject.toml encontrado
✅ Configuración completada



# 📘 Fase 2: Comprensión de los Datos (CRISP-DM)

Este notebook documenta la **segunda fase de CRISP-DM: Comprensión de los Datos**,  
enfocada en el **Análisis Exploratorio de Datos (EDA)**.

Se relaciona directamente con el pipeline de *EDA*, que incluye:

- Análisis de la distribución de rarezas en los mazos.  
- Identificación de las cartas más usadas.  
- Análisis del uso de win conditions.  
- Generación de un resumen exploratorio (EDA summary).


## 1. Distribución de rarezas en los mazos

Se analiza cómo se distribuyen las cartas por rareza (Común, Rara, Épica, Legendaria).  
Se busca responder:

- ¿Qué rarezas dominan los mazos?  
- ¿Existen diferencias entre días de la temporada?  


## 2. Cartas más usadas

Se identifican las cartas más populares en los mazos de los jugadores.  
Se busca responder:

- ¿Cuáles son las cartas más frecuentes en general?  
- ¿Hay cartas que ganan popularidad en ciertos días?  


## 3. Win Conditions más usadas

Las **win conditions** son cartas clave que suelen definir la estrategia del mazo.  
Se analiza su frecuencia y efectividad en diferentes días.  
Se busca responder:

- ¿Qué win conditions se usan más a lo largo de la temporada?  
- ¿Hay cambios de tendencia entre días?  


## 4. Resumen exploratorio (EDA Summary)

Se sintetizan los hallazgos principales de la fase exploratoria.  
Esto servirá de base para la fase de preparación de datos y potencial modelado.


In [2]:
# Cargar salidas del pipeline de EDA desde Kedro
# Nota: Esta celda requiere que la celda de configuración inicial se haya ejecutado

try:
    # Verificar que project_root está definido
    if 'project_root' not in globals():
        raise NameError("project_root no está definido. Ejecuta primero la celda de configuración inicial.")
    
    print("🔄 Inicializando Kedro...")
    
    # Importar y configurar Kedro
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project
    
    # Bootstrap del proyecto
    print("  - Bootstrap del proyecto...")
    metadata = bootstrap_project(project_root)
    print(f"  ✓ Proyecto: {metadata.project_name}")
    
    # Crear sesión de Kedro
    print("  - Creando sesión de Kedro...")
    session = KedroSession.create(project_path=project_root)
    context = session.load_context()
    
    # Obtener el catálogo
    catalog = context.catalog
    
    print("\n✅ Contexto de Kedro cargado exitosamente")
    
    # Cargar datos del catálogo
    print("\n📊 Cargando datos del pipeline de EDA...")
    
    analysis_data = {}
    for dataset_name in ["rarity_distributions_analysis", "most_used_cards_analysis", 
                         "win_conditions_usage_analysis", "eda_summary"]:
        try:
            data = catalog.load(dataset_name)
            analysis_data[dataset_name] = data
            print(f"✓ {dataset_name} cargado")
        except Exception as e:
            print(f"⚠ {dataset_name} no disponible: {str(e)[:80]}")
            analysis_data[dataset_name] = None
    
    # Asignar a variables
    rarity_analysis = analysis_data.get("rarity_distributions_analysis")
    most_used_cards = analysis_data.get("most_used_cards_analysis")
    win_conditions_usage = analysis_data.get("win_conditions_usage_analysis")
    eda_summary = analysis_data.get("eda_summary")
    
    # Mostrar resultados
    if rarity_analysis is not None:
        print("\n📊 Distribución de rarezas:")
        try:
            display(rarity_analysis)
        except:
            print(rarity_analysis)
    
    if most_used_cards is not None:
        print("\n🃏 Cartas más usadas:")
        try:
            display(most_used_cards)
        except:
            print(most_used_cards)
    
    if win_conditions_usage is not None:
        print("\n🎯 Uso de win conditions:")
        try:
            display(win_conditions_usage)
        except:
            print(win_conditions_usage)
    
    if eda_summary is not None:
        print("\n📋 Resumen del EDA:")
        try:
            display(eda_summary)
        except:
            print(eda_summary)
    
    print("\n💡 Nota: Si los datos no están disponibles, ejecuta primero:")
    print("   kedro run --pipeline=eda")
    
except NameError as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Ejecuta primero la celda de configuración inicial")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Posibles soluciones:")
    print("   1. Verifica que ejecutaste la celda de configuración inicial")
    print("   2. Verifica que pyproject.toml existe")
    print("   3. Si los datos no están disponibles, ejecuta: kedro run --pipeline=eda")
    import traceback
    traceback.print_exc()


🔄 Inicializando Kedro...


[11/28/25 16:46:36] INFO     Using 'conf\logging.yml' as logging configuration. You can change this __init__.py:269
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

  - Bootstrap del proyecto...
  ✓ Proyecto: ML ClashRoyale
  - Creando sesión de Kedro...


[11/28/25 16:46:41] WARNING  TensorFlow no está disponible. Los autoencoders no se    anomaly_detection_nodes.py:24
                             podrán usar.                                                                          

                    WARNING  c:\Users\PC                                                            warnings.py:112
                             RST\Documents\GitHub\ML_ClashRoyale\venv\Lib\site-packages\kedro\frame                
                             work\project\__init__.py:350: UserWarning: The                                        
                             'proyecto_ml_clashroyale.pipelines.nodes' module does not expose a                    
                             'create_pipeline' function, so no pipelines defined therein will be                   
                             returned by 'find_pipelines'.                                                         
                               warnings.warn(                                                                      
                                                                                                                   

...


🎯 Uso de win conditions:



{
    'winner_win_conditions': {
        26000021: {'card_name': 'Hog Rider', 'count': 1179956, 'percentage': 12.04},
        26000055: {'card_name': 'Mega Knight', 'count': 1118181, 'percentage': 11.41},
        26000015: {'card_name': 'Baby Dragon', 'count': 1077456, 'percentage': 10.99},
        28000004: {'card_name': 'Goblin Barrel', 'count': 920496, 'percentage': 9.39},
        26000006: {'card_name': 'Balloon', 'count': 835287, 'percentage': 8.52},
        26000016: {'card_name': 'Prince', 'count': 653367, 'percentage': 6.67},
        26000032: {'card_name': 'Miner', 'count': 526075, 'percentage': 5.37},
        26000020: {'card_name': 'Giant Skeleton', 'count': 440020, 'percentage': 4.49},
        26000009: {'card_name': 'Golem', 'count': 426026, 'percentage': 4.35},
        26000056: {'card_name': 'Skeleton Barrel', 'count': 384455, 'percentage': 3.92},
        26000024: {'card_name': 'Royal Giant', 'count': 351601, 'percentage': 3.59},
        26000003: {'card_name': 'Giant'


📋 Resumen del EDA:



{
    'eda_overview': {
        'dataset_size': 5644203,
        'analysis_focus': [
            'Cartas más usadas',
            'Win conditions más usadas',
            'Distribución de rarezas'
        ]
    },
    'key_findings': {
        'most_popular_cards': ['The Log', 'Wizard', 'Valkyrie', 'Zap', 'Skeleton Army'],
        'total_unique_cards': 102,
        'total_win_conditions': 24
    },
    'rarity_insights': {},
    'insights': [
        'Dataset contiene 5,644,203 registros de batallas',
        'Se identificaron 102 cartas únicas',
        'Total de win conditions analizadas: 24',
        'Carta más popular: The Log (3.59% de uso)',
        'Win condition más usada: Hog Rider (11.92% de uso)',
        'Perdedores usan más cartas common: +0.01 promedio',
        'Perdedores usan más cartas rare: +0.03 promedio',
        'Ganadores usan más cartas epic: +0.03 promedio',
        'Ganadores usan más cartas legendary: +0.01 promedio'
    ]
}


💡 Nota: Si los datos no están disponibles, ejecuta primero:
   kedro run --pipeline=eda
